# 🔗 ImageBind — CSC14005 Final Project
**Topic 17 | Nhóm 2 người | Focused type**

| Tuần | Nội dung |
|------|----------|
| Tuần 1 | Setup môi trường · Clone repo · Chạy demo · Chuẩn bị data |
| Tuần 2 | Training loop · Checkpoint · Evaluate · Viết metric |
| Tuần 3 | Benchmark hyperparams · Full evaluate · Vẽ biểu đồ |

> **Chạy trên Kaggle**: Accelerator → GPU T4 x2 · Internet → ON

---
# 📗 TUẦN 1 — Setup & Chuẩn bị

## 1.1 Cài đặt môi trường

In [ ]:
# Clone repo ImageBind
import os

if not os.path.exists('ImageBind'):
    !git clone https://github.com/facebookresearch/ImageBind.git
    print('Cloned!')
else:
    print('Repo da ton tai, bo qua.')

%cd ImageBind

# Cai dependencies
!pip install -q .
!pip install -q timm ftfy regex einops fvcore tqdm matplotlib seaborn pandas
print('\n=== Cai dat xong ===')

In [ ]:
import torch

print('=== Kiem tra moi truong ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nSu dung device  : {DEVICE}')

## 1.2 Tải pretrained weights

In [ ]:
import os

os.makedirs('.checkpoints', exist_ok=True)
WEIGHT_PATH = '.checkpoints/imagebind_huge.pth'

if not os.path.exists(WEIGHT_PATH):
    print('Dang tai pretrained weights (~2GB)...')
    !wget -q --show-progress -O {WEIGHT_PATH} \
        https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth
    print('Tai xong!')
else:
    size_mb = os.path.getsize(WEIGHT_PATH) / 1e6
    print(f'Weights da co san: {size_mb:.0f} MB')

## 1.3 Load model & Smoke test (demo pipeline)

In [ ]:
from imagebind import data as ibdata
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

print('Dang load model...')
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(DEVICE)
print('Load model thanh cong!')

# Kiem tra so luong params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTong params      : {total_params / 1e6:.1f}M')
print(f'Trainable params : {trainable_params / 1e6:.1f}M')

In [ ]:
# Smoke test: thu text embedding
import torch

texts = [
    'A dog is barking',
    'Fire crackling sound',
    'Ocean waves crashing',
    'Rain falling on roof',
]

inputs = {
    ModalityType.TEXT: ibdata.load_and_transform_text(texts, DEVICE)
}

with torch.no_grad():
    embeddings = model(inputs)

emb = embeddings[ModalityType.TEXT]
print(f'Text embedding shape: {emb.shape}')  # Expected: [4, 1024]
print(f'Embedding norm      : {emb.norm(dim=-1)}')  # Should be ~1.0 (normalized)

# Cosine similarity giua cac text
sim = emb @ emb.T
print(f'\nCosine similarity matrix:')
import pandas as pd
labels = ['dog', 'fire', 'ocean', 'rain']
print(pd.DataFrame(sim.cpu().numpy().round(3), index=labels, columns=labels))
print('\n=== Smoke test PASS ===')

## 1.4 Chuẩn bị dataset — ESC-50 (Audio Classification)

In [ ]:
# Download ESC-50
import os

if not os.path.exists('ESC-50'):
    print('Dang tai ESC-50...')
    !git clone https://github.com/karolpiczak/ESC-50.git
    print('Tai xong!')
else:
    print('ESC-50 da co san.')

# Kiem tra cau truc
import glob
audio_files = glob.glob('ESC-50/audio/*.wav')
print(f'\nSo file audio: {len(audio_files)}')
print(f'Vi du         : {audio_files[0]}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Doc metadata
meta = pd.read_csv('ESC-50/meta/esc50.csv')
print('=== ESC-50 Dataset Info ===')
print(meta.head(5))
print(f'\nTong so mau  : {len(meta)}')
print(f'So class     : {meta["category"].nunique()}')
print(f'So fold      : {meta["fold"].nunique()}')
print(f'\nCac class (10 dau):')
print(meta['category'].value_counts().head(10))

# Plot phan phoi class
fig, ax = plt.subplots(figsize=(14, 4))
meta['category'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('ESC-50: So mau theo class (moi class = 40 mau)')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('esc50_distribution.png', dpi=100)
plt.show()
print('Luu bieu do: esc50_distribution.png')

In [ ]:
import os
import torch
import torchaudio
import numpy as np
from torch.utils.data import Dataset, DataLoader
import pandas as pd

class ESC50Dataset(Dataset):
    """
    ESC-50 dataset wrapper cho ImageBind.
    Tra ve (mel_spectrogram, label) cho moi audio clip.
    """
    def __init__(self, meta_path, audio_dir, folds, target_sr=16000, duration=2.0):
        self.meta = pd.read_csv(meta_path)
        self.meta = self.meta[self.meta['fold'].isin(folds)].reset_index(drop=True)
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = int(target_sr * duration)
        
        # Map category -> int label
        categories = sorted(self.meta['category'].unique())
        self.class2idx = {c: i for i, c in enumerate(categories)}
        self.idx2class = {i: c for c, i in self.class2idx.items()}
        self.class_names = categories

    def __len__(self):
        return len(self.meta)

    def _load_audio(self, filepath):
        waveform, sr = torchaudio.load(filepath)
        # Resample neu can
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # Pad hoac cat cho du duration
        if waveform.shape[1] < self.num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.num_samples - waveform.shape[1]))
        else:
            waveform = waveform[:, :self.num_samples]
        return waveform

    def _waveform_to_melspec(self, waveform):
        """Chuyen waveform -> mel-spectrogram (128 bins) theo paper ImageBind"""
        mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.target_sr,
            n_fft=400,
            win_length=400,      # 25ms window
            hop_length=160,      # 10ms hop
            n_mels=128,
        )
        mel = mel_transform(waveform)             # (1, 128, T)
        mel = torchaudio.transforms.AmplitudeToDB()(mel)
        mel = mel.squeeze(0)                      # (128, T)
        # Normalize
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        filepath = os.path.join(self.audio_dir, row['filename'])
        waveform = self._load_audio(filepath)
        mel = self._waveform_to_melspec(waveform)
        label = self.class2idx[row['category']]
        return mel, label


# Khoi tao dataset
# Fold 1-4: train | Fold 5: test (theo chuan ESC-50)
train_dataset = ESC50Dataset(
    meta_path='ESC-50/meta/esc50.csv',
    audio_dir='ESC-50/audio',
    folds=[1, 2, 3, 4]
)
test_dataset = ESC50Dataset(
    meta_path='ESC-50/meta/esc50.csv',
    audio_dir='ESC-50/audio',
    folds=[5]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f'Train samples : {len(train_dataset)}')
print(f'Test  samples : {len(test_dataset)}')
print(f'So classes    : {len(train_dataset.class_names)}')
print(f'Class names   : {train_dataset.class_names[:5]}...')

# Kiem tra 1 batch
mel, label = next(iter(train_loader))
print(f'\nBatch mel shape : {mel.shape}')
print(f'Batch label     : {label[:5]}')
print('=== Dataset OK ===')

---
# 📘 TUẦN 2 — Training Loop & Evaluate

## 2.1 Cấu hình training

In [ ]:
# ============================================================
# CAU HINH TRAINING — chinh sua o day
# ============================================================
CONFIG = {
    # Training
    'num_epochs'    : 2,          # >= 1 de dat muc Mini
    'batch_size'    : 32,
    'lr'            : 1.6e-3,
    'weight_decay'  : 0.2,
    'grad_clip'     : 1.0,
    'temperature'   : 0.05,       # tau cho InfoNCE loss

    # Checkpoint
    'ckpt_dir'      : 'checkpoints',
    'log_interval'  : 5,          # log moi N steps

    # Freeze
    'freeze_vision' : True,       # Freeze vision encoder (theo paper)
    'freeze_text'   : True,       # Freeze text encoder
}

import os
os.makedirs(CONFIG['ckpt_dir'], exist_ok=True)
os.makedirs('results', exist_ok=True)

print('Config:')
for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')

## 2.2 Setup model để train

In [ ]:
from imagebind.models import imagebind_model

# Load model moi hoan toan de train
train_model = imagebind_model.imagebind_huge(pretrained=True)
train_model.to(DEVICE)

# Freeze encoder theo config
frozen, trainable = 0, 0
for name, param in train_model.named_parameters():
    should_freeze = (
        (CONFIG['freeze_vision'] and 'modality_trunks.vision' in name) or
        (CONFIG['freeze_text']   and 'modality_trunks.text'   in name) or
        ('modality_postprocessors.vision' in name) or
        ('modality_postprocessors.text'   in name)
    )
    param.requires_grad = not should_freeze
    if should_freeze:
        frozen    += param.numel()
    else:
        trainable += param.numel()

print(f'Frozen params    : {frozen    / 1e6:.1f}M')
print(f'Trainable params : {trainable / 1e6:.1f}M')

# Optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, train_model.parameters()),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.95),
)

# LR scheduler (cosine)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['num_epochs'],
    eta_min=1e-5,
)

print('Optimizer va Scheduler da san sang.')

## 2.3 InfoNCE Loss

In [ ]:
import torch
import torch.nn.functional as F

def infonce_loss(emb_i, emb_m, temperature):
    """
    Symmetric InfoNCE loss (Equation 1 trong paper ImageBind).

    Args:
        emb_i : Tensor (B, D) - image embedding
        emb_m : Tensor (B, D) - modality embedding  
        temperature: float - tau

    Returns:
        loss : scalar
    """
    # L2 normalize
    q = F.normalize(emb_i, dim=-1)   # (B, D)
    k = F.normalize(emb_m, dim=-1)   # (B, D)

    # Logits: (B, B) - moi cap (i, j)
    logits = q @ k.T / temperature

    # Labels: diagonal la positive
    labels = torch.arange(len(q), device=q.device)

    # L(I->M) + L(M->I)
    loss_im = F.cross_entropy(logits,   labels)
    loss_mi = F.cross_entropy(logits.T, labels)

    return (loss_im + loss_mi) / 2


# Test loss voi random tensors
q_test = torch.randn(8, 1024)
k_test = torch.randn(8, 1024)
loss_test = infonce_loss(q_test, k_test, temperature=0.05)
print(f'Test loss (random input, ky vong ~log(8)={torch.log(torch.tensor(8.)):.3f}): {loss_test:.3f}')
print('InfoNCE loss OK!')

## 2.4 Training Loop chính

In [ ]:
import json
import time
from tqdm import tqdm
import torch

def train_one_epoch(model, loader, optimizer, epoch, config, class_names):
    """
    Train 1 epoch tren ESC-50.
    Strategy: dung text embedding cua class name lam 'image proxy'
    (vi ta khong co anh tuong ung voi audio trong ESC-50)
    => align (text_class, audio) thay vi (image, audio)
    """
    model.train()
    total_loss = 0
    start = time.time()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=True)
    for step, (mel_batch, label_batch) in enumerate(pbar):
        mel_batch   = mel_batch.to(DEVICE)    # (B, 128, T)
        label_batch = label_batch.to(DEVICE)

        # Lay text embedding cua class name tuong ung
        batch_class_names = [class_names[l.item()] for l in label_batch]
        with torch.no_grad():
            text_inputs = {
                ModalityType.TEXT: ibdata.load_and_transform_text(
                    batch_class_names, DEVICE
                )
            }
            text_emb = model(text_inputs)[ModalityType.TEXT]  # (B, 1024)

        # Audio embedding — phan nay duoc update
        # Reshape mel -> phu hop voi ImageBind audio input
        # ImageBind can: (B, 1, 128, T)
        mel_input = mel_batch.unsqueeze(1)    # (B, 1, 128, T)

        audio_inputs = {
            ModalityType.AUDIO: mel_input
        }
        audio_emb = model(audio_inputs)[ModalityType.AUDIO]   # (B, 1024)

        # Tinh loss
        loss = infonce_loss(text_emb, audio_emb, config['temperature'])

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), config['grad_clip']
        )
        optimizer.step()

        total_loss += loss.item()

        if step % config['log_interval'] == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(loader)
    elapsed  = time.time() - start
    print(f'  Epoch {epoch} | Avg Loss: {avg_loss:.4f} | Time: {elapsed:.0f}s')
    return avg_loss


# ============================================================
# CHAY TRAINING
# ============================================================
training_log = []

print(f'=== Bat dau train {CONFIG["num_epochs"]} epoch(s) ===')

for epoch in range(1, CONFIG['num_epochs'] + 1):
    avg_loss = train_one_epoch(
        model=train_model,
        loader=train_loader,
        optimizer=optimizer,
        epoch=epoch,
        config=CONFIG,
        class_names=train_dataset.class_names,
    )
    scheduler.step()

    # Log
    training_log.append({
        'epoch'   : epoch,
        'loss'    : round(avg_loss, 6),
        'lr'      : scheduler.get_last_lr()[0],
        'temperature': CONFIG['temperature'],
    })

    # Luu checkpoint
    ckpt_path = f'{CONFIG["ckpt_dir"]}/epoch_{epoch:02d}_loss{avg_loss:.4f}.pth'
    torch.save({
        'epoch'          : epoch,
        'model_state'    : train_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss'           : avg_loss,
        'config'         : CONFIG,
    }, ckpt_path)
    print(f'  Checkpoint saved: {ckpt_path}')

# Luu log
with open('results/training_log.json', 'w') as f:
    json.dump(training_log, f, indent=2)

print('\n=== Training hoan tat ===')
print(f'Log da luu: results/training_log.json')

## 2.5 Vẽ loss curve

In [ ]:
import matplotlib.pyplot as plt
import json

with open('results/training_log.json') as f:
    log = json.load(f)

epochs = [x['epoch'] for x in log]
losses = [x['loss']  for x in log]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, losses, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_title('Training Loss — ImageBind (ESC-50, InfoNCE)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
for e, l in zip(epochs, losses):
    ax.annotate(f'{l:.4f}', (e, l), textcoords='offset points', xytext=(0, 8), ha='center')
plt.tight_layout()
plt.savefig('results/loss_curve.png', dpi=120)
plt.show()
print('Luu: results/loss_curve.png')

## 2.6 Evaluate — Zero-shot Accuracy trên test split

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

@torch.no_grad()
def zero_shot_accuracy(model, test_loader, class_names, device, temperature=0.05):
    """
    Zero-shot audio classification:
    1. Encode tat ca class names -> text embeddings
    2. Encode tung audio clip -> audio embedding
    3. Classify bang cosine similarity
    """
    model.eval()

    # Buoc 1: Encode class names
    # Dung template: "This is a sound of {class}"
    prompts = [f'This is a sound of {c.replace("_", " ")}' for c in class_names]
    text_inputs = {
        ModalityType.TEXT: ibdata.load_and_transform_text(prompts, device)
    }
    text_emb = model(text_inputs)[ModalityType.TEXT]          # (C, D)
    text_emb = F.normalize(text_emb, dim=-1)

    # Buoc 2: Encode audio va classify
    all_preds, all_labels = [], []

    for mel_batch, label_batch in tqdm(test_loader, desc='Evaluating'):
        mel_input = mel_batch.unsqueeze(1).to(device)         # (B, 1, 128, T)
        audio_inputs = {ModalityType.AUDIO: mel_input}
        audio_emb = model(audio_inputs)[ModalityType.AUDIO]   # (B, D)
        audio_emb = F.normalize(audio_emb, dim=-1)

        # Cosine similarity -> softmax -> argmax
        sim   = audio_emb @ text_emb.T / temperature          # (B, C)
        preds = sim.argmax(dim=-1).cpu()

        all_preds.append(preds)
        all_labels.append(label_batch)

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    # Top-1 accuracy
    top1 = (all_preds == all_labels).float().mean().item() * 100

    # Per-class accuracy
    per_class = {}
    for cls_idx, cls_name in enumerate(class_names):
        mask = (all_labels == cls_idx)
        if mask.sum() > 0:
            per_class[cls_name] = (all_preds[mask] == all_labels[mask]).float().mean().item() * 100

    return top1, all_preds, all_labels, per_class


# Chay evaluate
print('=== Zero-shot Evaluation tren test split (Fold 5) ===')
top1, preds, labels, per_class = zero_shot_accuracy(
    model=train_model,
    test_loader=test_loader,
    class_names=test_dataset.class_names,
    device=DEVICE,
    temperature=CONFIG['temperature'],
)
print(f'\nTop-1 Accuracy: {top1:.2f}%')
print(f'(Baseline tu paper: ImageBind pretrained ~66.9% tren ESC-50 5-fold avg)')

# Luu ket qua
import json
result = {
    'top1_accuracy'   : round(top1, 4),
    'num_test_samples': len(test_dataset),
    'temperature'     : CONFIG['temperature'],
    'per_class_acc'   : {k: round(v, 4) for k, v in per_class.items()},
}
with open('results/eval_results.json', 'w') as f:
    json.dump(result, f, indent=2)
print('Luu: results/eval_results.json')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Confusion matrix (top 10 class de doc)
top10_classes = sorted(per_class, key=per_class.get)[:10]  # 10 class kem nhat
top10_idx     = [test_dataset.class2idx[c] for c in top10_classes]

mask = torch.isin(labels, torch.tensor(top10_idx))
cm = confusion_matrix(
    labels[mask].numpy(),
    preds[mask].numpy(),
    labels=top10_idx
)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top10_classes, yticklabels=top10_classes, ax=ax)
ax.set_title('Confusion Matrix — 10 Class kem nhat (normalized)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=120)
plt.show()
print('Luu: results/confusion_matrix.png')

---
# 📙 TUẦN 3 — Benchmark Hyperparameters

## 3.1 Benchmark Temperature τ
> Theo ablation study trong paper: tau anh huong manh den ket qua

In [ ]:
import json
import torch

# Load checkpoint tot nhat tu tuan 2
import glob
ckpt_files = sorted(glob.glob('checkpoints/*.pth'))
latest_ckpt = ckpt_files[-1]
print(f'Dung checkpoint: {latest_ckpt}')

def load_model_from_ckpt(ckpt_path):
    from imagebind.models import imagebind_model
    m = imagebind_model.imagebind_huge(pretrained=False)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    m.to(DEVICE)
    return m


# Benchmark nhieu gia tri temperature
temperatures = [0.01, 0.05, 0.07, 0.1, 0.2, 0.5, 1.0]
temp_results = {}

eval_model = load_model_from_ckpt(latest_ckpt)

for tau in temperatures:
    top1, _, _, _ = zero_shot_accuracy(
        model=eval_model,
        test_loader=test_loader,
        class_names=test_dataset.class_names,
        device=DEVICE,
        temperature=tau,
    )
    temp_results[tau] = round(top1, 4)
    print(f'  tau={tau:.3f} -> Top-1: {top1:.2f}%')

with open('results/benchmark_temperature.json', 'w') as f:
    json.dump(temp_results, f, indent=2)
print('Luu: results/benchmark_temperature.json')

## 3.2 Benchmark Epochs (so sanh 1 vs 2 vs ... epoch)

In [ ]:
# So sanh ket qua evaluate sau moi epoch
epoch_results = {}

for ckpt_path in sorted(glob.glob('checkpoints/*.pth')):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    epoch_num = ckpt['epoch']
    
    m = load_model_from_ckpt(ckpt_path)
    top1, _, _, _ = zero_shot_accuracy(
        model=m,
        test_loader=test_loader,
        class_names=test_dataset.class_names,
        device=DEVICE,
        temperature=CONFIG['temperature'],
    )
    epoch_results[epoch_num] = round(top1, 4)
    print(f'  Epoch {epoch_num} -> Top-1: {top1:.2f}%')
    del m
    torch.cuda.empty_cache()

with open('results/benchmark_epochs.json', 'w') as f:
    json.dump(epoch_results, f, indent=2)
print('Luu: results/benchmark_epochs.json')

## 3.3 Benchmark Batch Size

In [ ]:
# Chay 1 epoch ngan voi cac batch size khac nhau
# (Chi chay 20 steps de so sanh toc do hoi tu)
from torch.utils.data import DataLoader

batch_sizes   = [16, 32, 64]
bs_results    = {}
MAX_STEPS     = 20   # chi lay 20 steps mau de nhanh

for bs in batch_sizes:
    loader_bs = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=2)
    m = load_model_from_ckpt(latest_ckpt)
    m.train()
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, m.parameters()),
        lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
    )
    losses = []
    for step, (mel, lbl) in enumerate(loader_bs):
        if step >= MAX_STEPS:
            break
        mel    = mel.unsqueeze(1).to(DEVICE)
        names  = [train_dataset.class_names[l.item()] for l in lbl]
        with torch.no_grad():
            t_emb = m({ModalityType.TEXT: ibdata.load_and_transform_text(names, DEVICE)})[ModalityType.TEXT]
        a_emb = m({ModalityType.AUDIO: mel})[ModalityType.AUDIO]
        loss  = infonce_loss(t_emb, a_emb, CONFIG['temperature'])
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    bs_results[bs] = {
        'avg_loss_first20': round(float(np.mean(losses)), 6),
        'final_loss'       : round(losses[-1], 6),
    }
    print(f'  batch_size={bs:3d} | avg_loss={np.mean(losses):.4f} | final={losses[-1]:.4f}')
    del m
    torch.cuda.empty_cache()

with open('results/benchmark_batchsize.json', 'w') as f:
    json.dump(bs_results, f, indent=2)
print('Luu: results/benchmark_batchsize.json')

## 3.4 Tổng hợp và vẽ biểu đồ Benchmark

In [ ]:
import matplotlib.pyplot as plt
import json
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ImageBind — Benchmark Results (ESC-50)', fontsize=14, fontweight='bold')

# --- Plot 1: Temperature ---
with open('results/benchmark_temperature.json') as f:
    temp_data = json.load(f)
taus = [float(k) for k in temp_data]
accs = list(temp_data.values())
best_tau = taus[np.argmax(accs)]
axes[0].bar([str(t) for t in taus], accs, color='steelblue', alpha=0.8)
axes[0].axvline(x=taus.index(best_tau), color='red', linestyle='--', linewidth=1.5, label=f'Best tau={best_tau}')
axes[0].set_title('Temperature tau vs Top-1 Acc')
axes[0].set_xlabel('Temperature (tau)')
axes[0].set_ylabel('Top-1 Accuracy (%)')
axes[0].legend(fontsize=9)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=8)

# --- Plot 2: Epochs ---
with open('results/benchmark_epochs.json') as f:
    epoch_data = json.load(f)
ep_keys = [int(k) for k in epoch_data]
ep_vals = list(epoch_data.values())
axes[1].plot(ep_keys, ep_vals, 'o-', color='darkorange', linewidth=2, markersize=8)
axes[1].set_title('Epoch vs Top-1 Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Top-1 Accuracy (%)')
axes[1].grid(True, alpha=0.3)
for e, v in zip(ep_keys, ep_vals):
    axes[1].annotate(f'{v:.1f}%', (e, v), xytext=(0, 8), textcoords='offset points', ha='center')

# --- Plot 3: Batch size ---
with open('results/benchmark_batchsize.json') as f:
    bs_data = json.load(f)
bs_keys = [str(k) for k in bs_data]
bs_loss = [bs_data[k]['avg_loss_first20'] for k in bs_data]
axes[2].bar(bs_keys, bs_loss, color='mediumseagreen', alpha=0.8)
axes[2].set_title('Batch Size vs Avg Loss (20 steps)')
axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('Avg Loss')
for i, v in enumerate(bs_loss):
    axes[2].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/benchmark_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('Luu: results/benchmark_summary.png')

## 3.5 Bảng tổng hợp kết quả cuối cùng

In [ ]:
import pandas as pd
import json

print('=' * 60)
print('  KET QUA TONG HOP — ImageBind Project')
print('=' * 60)

# Training summary
with open('results/training_log.json') as f:
    log = json.load(f)
print('\n[Training]')
df_train = pd.DataFrame(log)
print(df_train.to_string(index=False))

# Eval summary
with open('results/eval_results.json') as f:
    ev = json.load(f)
print(f'\n[Evaluate - Zero-shot Top-1 Accuracy]')
print(f'  Ket qua nhom : {ev["top1_accuracy"]:.2f}%')
print(f'  Paper (pretrained, 5-fold avg) : 66.9%')
print(f'  AudioCLIP (co train audio-text): 68.6%')

# Temperature benchmark
with open('results/benchmark_temperature.json') as f:
    td = json.load(f)
best_tau = max(td, key=td.get)
print(f'\n[Benchmark Temperature]')
print(f'  Best tau = {best_tau} -> {td[best_tau]:.2f}%')

# Top/Bottom class
sorted_cls = sorted(ev['per_class_acc'].items(), key=lambda x: x[1])
print(f'\n[Per-class Analysis]')
print('  5 class kem nhat:')
for cls, acc in sorted_cls[:5]:
    print(f'    {cls:25s}: {acc:.1f}%')
print('  5 class tot nhat:')
for cls, acc in sorted_cls[-5:]:
    print(f'    {cls:25s}: {acc:.1f}%')

print('\n' + '=' * 60)
print('  Tat ca ket qua luu trong thu muc: results/')
print('=' * 60)

In [ ]:
# Liet ke tat ca file ket qua
import os
print('=== Files da tao ===')
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path:45s} ({size:,} bytes)')

for root, dirs, files in os.walk('checkpoints'):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path:45s} ({size / 1e6:.0f} MB)')